# 02 - Hybrid RAG Walkthrough

**Lecture goal:** show how Hybrid RAG combines semantic vector retrieval with exact-token BM25 retrieval, then fuses and reranks results before asking the LLM to answer.

**Use case:** the same AI-labs corpus used in the KG-RAG notebook. This notebook focuses on questions where exact terms, acronyms, paper titles, and paraphrases all matter.

**Notebook sections:**
1. Build the hybrid index - create dense and sparse views of the same corpus
2. Inspect dense vs. sparse retrieval - see why each retriever helps
3. Run the full pipeline - ask questions and view input/output examples
4. Ablate the components - reason about dense-only, sparse-only, and reranking
5. Discussion - when Hybrid RAG is the right default

The saved cell outputs are lecture examples. Your exact answers can vary because the final response is generated by an LLM.


## 1. Build the Hybrid Index - One Corpus, Two Retrieval Views

Hybrid RAG indexes the same chunks in two ways:

```text
Documents -> chunks -> dense embeddings -> vector store / Qdrant
Documents -> chunks -> sparse terms     -> BM25 index
```

At query time, both retrievers run. The pipeline fuses their ranked lists with Reciprocal Rank Fusion, then uses a cross-encoder reranker to improve the final ordering before generation.


In [1]:
from rag_compare.common import load_corpus
from rag_compare.hybrid_rag import HybridRagPipeline

docs = load_corpus()
print(f"Loaded {len(docs)} source documents")
print("Example document preview:")
print(docs[0].text[:300].replace("\n", " ") + "...")

hybrid = HybridRagPipeline.build(docs)
print("Hybrid RAG query engine is ready")


Loaded 18 source documents
Example document preview:
# Large language model  A large language model (LLM) is a language model notable for its ability to achieve general-purpose language generation and other natural language processing tasks...
Hybrid RAG query engine is ready


### What happened in section 1?

1. `load_corpus()` loaded the shared document set.
2. `HybridRagPipeline.build(docs)` split documents into chunks.
3. Dense retrieval stored embeddings in Qdrant.
4. Sparse retrieval built a BM25 index over the same chunks.
5. `QueryFusionRetriever` combined dense and sparse ranked lists.
6. `SentenceTransformerRerank` reranked the fused candidates before the LLM answered.


## 2. Inspect Dense vs. Sparse Retrieval - Why Hybrid Helps

Dense and sparse retrieval fail in different ways.

- **Dense retrieval** is good for meaning and paraphrase.
- **Sparse retrieval / BM25** is good for exact words, acronyms, names, years, and paper titles.
- **Hybrid retrieval** keeps both signals so one retriever can recover what the other misses.


In [2]:
retrieval_examples = [
    {
        "label": "exact-term query",
        "question": (
            "What is the exact title of the 2017 paper that introduced "
            "the Transformer architecture?"
        ),
        "dense_strength": "understands the meaning of 'paper that introduced Transformer'",
        "sparse_strength": "matches rare exact terms like '2017', 'paper', and 'Transformer'",
    },
    {
        "label": "semantic/paraphrase query",
        "question": "What technique uses human preference feedback to train language models?",
        "dense_strength": "matches the meaning of RLHF even if the acronym is not used",
        "sparse_strength": "helps if source chunks contain words like 'feedback' or 'human'",
    },
]

for example in retrieval_examples:
    print("=" * 80)
    print(f"Query type: {example['label']}")
    print(f"Input: {example['question']}")
    print(f"Dense retriever contribution: {example['dense_strength']}")
    print(f"Sparse retriever contribution: {example['sparse_strength']}")


Query type: exact-term query
Input: What is the exact title of the 2017 paper that introduced the Transformer architecture?
Dense retriever contribution: understands the meaning of 'paper that introduced Transformer'
Sparse retriever contribution: matches rare exact terms like '2017', 'paper', and 'Transformer'
Query type: semantic/paraphrase query
Input: What technique uses human preference feedback to train language models?
Dense retriever contribution: matches the meaning of RLHF even if the acronym is not used
Sparse retriever contribution: helps if source chunks contain words like 'feedback' or 'human'


### Retrieval flow in this repo

The implementation in `src/rag_compare/hybrid_rag/pipeline.py` follows this path:

```text
question
  -> dense retriever top-k from Qdrant
  -> sparse retriever top-k from BM25
  -> reciprocal-rank fusion
  -> cross-encoder rerank
  -> LLM answer
```

For students: the point is not that dense or sparse is always better. The point is that they are complementary.


## 3. Run the Full Pipeline - Inputs and Outputs

The input to Hybrid RAG is a natural-language question. The output is a generated answer grounded in the fused and reranked retrieved chunks.

The first example rewards exact-token retrieval because the answer is a precise paper title. The second rewards both exact acronym matching and semantic context.


In [3]:
questions = [
    (
        "exact-term question",
        "What is the exact title of the 2017 paper that introduced the Transformer architecture?",
    ),
    (
        "acronym / definition question",
        "What does the acronym RLHF stand for?",
    ),
]

for label, question in questions:
    print("=" * 80)
    print(f"Example type: {label}")
    print(f"Hybrid RAG input question: {question}")
    print("Hybrid RAG output answer:")
    print(hybrid.query(question))
    print()


Example type: exact-term question
Hybrid RAG input question: What is the exact title of the 2017 paper that introduced the Transformer architecture?
Hybrid RAG output answer:
The exact title is "Attention Is All You Need."

Example type: acronym / definition question
Hybrid RAG input question: What does the acronym RLHF stand for?
Hybrid RAG output answer:
RLHF stands for Reinforcement Learning from Human Feedback.



### Input/output summary for lecture

| Stage | Example |
|---|---|
| Source document text | Wikipedia-derived pages about AI labs, models, papers, and training methods |
| Dense signal | Semantic similarity to `paper that introduced Transformer` |
| Sparse signal | Exact terms like `2017`, `Transformer`, `RLHF`, and paper titles |
| Fusion method | Reciprocal Rank Fusion combines dense and sparse rankings |
| Reranking method | Cross-encoder scores final query/chunk pairs |
| User question | `What does the acronym RLHF stand for?` |
| Final answer | `Reinforcement Learning from Human Feedback` |


## 4. Ablate the Components - Where Each Part Earns Its Keep

A good lecture exercise is to ask what happens when one part is removed. The full repo does not yet package separate dense-only and sparse-only pipeline classes, but this table gives students the expected comparison to test.


In [4]:
import pandas as pd

ablation_summary = pd.DataFrame(
    [
        {
            "variant": "dense only",
            "keeps": "embedding similarity",
            "removes": "BM25 exact-token matching + fusion",
            "expected weakness": "can miss exact titles, IDs, acronyms, and rare names",
        },
        {
            "variant": "sparse only",
            "keeps": "BM25 exact-token matching",
            "removes": "semantic embedding similarity + fusion",
            "expected weakness": "can miss paraphrases and conceptually similar wording",
        },
        {
            "variant": "fusion, no rerank",
            "keeps": "dense + sparse reciprocal-rank fusion",
            "removes": "cross-encoder reranking",
            "expected weakness": "top results can be less precisely ordered",
        },
        {
            "variant": "full hybrid",
            "keeps": "dense + sparse + fusion + rerank",
            "removes": "nothing",
            "expected weakness": "more moving parts than baseline RAG",
        },
    ]
)

ablation_summary


           variant                              keeps  ...              expected weakness
0       dense only               embedding similarity  ...  can miss exact titles, IDs, acronyms, and rare names
1      sparse only             BM25 exact-token matching  ...  can miss paraphrases and conceptually similar wording
2  fusion, no rerank  dense + sparse reciprocal-rank fusion  ...  top results can be less precisely ordered
3      full hybrid     dense + sparse + fusion + rerank  ...  more moving parts than baseline RAG

[4 rows x 4 columns]

### Suggested ablation exercise

Ask students to run the same questions through these variants:

1. Dense-only retriever.
2. BM25-only retriever.
3. Dense + sparse fusion without reranking.
4. Full Hybrid RAG.

Expected pattern:

- BM25 usually helps most on exact terms, IDs, titles, acronyms, and rare names.
- Dense retrieval usually helps most on paraphrase and semantic similarity.
- Fusion improves recall by combining both lists.
- Reranking improves precision by reordering the final candidates.


## 5. Discussion - When Hybrid RAG Is the Right Default

### What Hybrid RAG is good at

- General-purpose document Q&A.
- Corpora with both semantic questions and exact terms.
- Product docs, support docs, code search, internal wikis, and research corpora.
- Improving baseline RAG quality without building a knowledge graph.

### What can go wrong

- BM25 can overweight repeated or noisy terms.
- Dense retrieval can retrieve semantically related but factually wrong chunks.
- Rerankers add model downloads, latency, and another dependency.
- Hybrid RAG still does not truly perform graph-style multi-hop joins.

### Questions for students

1. Which example benefits more from BM25: the Transformer paper title or the RLHF definition?
2. What kinds of user questions would dense retrieval handle better than BM25?
3. Why does fusion improve recall?
4. Why does reranking improve precision?
5. When would you choose Hybrid RAG over KG-RAG?
